# 予測期間の比較

歴史的な実験コードです。現在の実行入口は `../09_confidence_nested.ipynb`。
元Notebookのセル番号は0始まりです。コードの個人フォルダ名は置換しています。独立実行は保証しません。
保存出力は `../../results/imported_20260907/`、監査は `../../docs/CONFIDENCE_AUDIT.md` を参照してください。


## 元のセル index 24


In [ ]:
# ============================================================
# FORECAST HORIZON COMPARISON
# 5分 ～ 120分
#
# 目的
# ------------------------------------------------------------
# 「30分先予測が本当に強いのか？」を純粋に検証する。
#
# 今回はあえて
# ・TP / SLなし
# ・Confidence sizingなし
# ・Probability calibrationなし
#
# とする。
#
# 理由:
# まず予測ホライズンそのもののedgeを比較したいから。
#
# 各ホライズン:
# 1. 過去データでモデル学習
# 2. Validationでconfidence thresholdだけ選択
# 3. Testでは設定固定
# 4. 時間決済のみ
#
# 必要:
# 前のセルで以下が定義済み
#
# load_bars
# make_features
# direction_features
# make_time_weights
# TRADING_COST
# HALF_LIFE_DAYS
#
# ============================================================


from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, accuracy_score


# ============================================================
# 1. 実験設定
# ============================================================

HORIZONS_MINUTES = [
    5,
    10,
    15,
    20,
    30,
    45,
    60,
    90,
    120,
]

# 5分足なので、
# 30分なら6本先
BAR_MINUTES = 5


# Validationで試すDirection confidence
CONFIDENCE_THRESHOLDS = [
    0.50,
    0.52,
    0.54,
    0.56,
    0.58,
    0.60,
    0.62,
    0.65,
    0.70,
]

OUTER_SPLITS_HORIZON = 5

MIN_VALIDATION_TRADES_HORIZON = 20

START_CAPITAL_HORIZON = 10000

RANDOM_STATE = 42


# ============================================================
# 2. 最新のUSD/JPY CSVを探す
# ============================================================

def find_latest_fx_snapshot():

    root = (
        Path.cwd()
        / "fx_experiment_runs"
    )

    files = list(
        root.glob(
            "*/usdjpy_5m.csv"
        )
    )

    if not files:

        raise FileNotFoundError(
            "fx_experiment_runs内に "
            "usdjpy_5m.csv がありません。"
        )

    return max(
        files,
        key=lambda p:
            p.stat().st_mtime
    )


csv_path = find_latest_fx_snapshot()

print(
    "使用CSV:"
)

print(
    csv_path
)


# ============================================================
# 3. 価格データ
# ============================================================

bars = load_bars(
    csv_path
)

print()
print(
    "5分足:",
    len(bars)
)


# ============================================================
# 4. Horizonごとの教師データ
# ============================================================

def prepare_horizon_data(
    bars,
    horizon_minutes,
):

    """
    シグナル足 t
         ↓
    次足OpenでEntry
         ↓
    horizon_minutes後のCloseでExit

    例:
    30分 → horizon_bars = 6
    """

    horizon_bars = (
        horizon_minutes
        // BAR_MINUTES
    )

    if (
        horizon_minutes
        % BAR_MINUTES
        != 0
    ):

        raise ValueError(
            "Horizonは5分単位にしてください"
        )

    df = make_features(
        bars
    )

    df[
        "entry_price"
    ] = (
        bars[
            "Open"
        ]
        .shift(-1)
    )

    df[
        "exit_price"
    ] = (
        bars[
            "Close"
        ]
        .shift(
            -horizon_bars
        )
    )

    df[
        "future_return"
    ] = (
        df[
            "exit_price"
        ]
        /
        df[
            "entry_price"
        ]
        - 1
    )

    # ----------------------------
    # Direction教師ラベル
    # ----------------------------

    df[
        "direction_target"
    ] = (
        df[
            "future_return"
        ]
        > 0
    ).astype(int)

    df[
        "horizon_bars"
    ] = (
        horizon_bars
    )

    # ----------------------------
    # ラベル終了時刻
    # Walk-Forward leakage防止用
    # ----------------------------

    times = pd.Series(
        bars.index,
        index=bars.index
    )

    df[
        "label_end"
    ] = (
        times.shift(
            -horizon_bars
        )
        +
        pd.Timedelta(
            minutes=BAR_MINUTES
        )
    )

    # ----------------------------
    # 途中に時間欠損がないか
    # ----------------------------

    complete = (
        times.shift(
            -horizon_bars
        )
        -
        times
    ).eq(
        pd.Timedelta(
            minutes=
                horizon_minutes
        )
    )

    required = (
        direction_features
        +
        [
            "future_return",
            "entry_price",
            "exit_price",
            "label_end",
        ]
    )

    data = (
        df.loc[
            complete
        ]
        .dropna(
            subset=required
        )
        .copy()
    )

    return data


# ============================================================
# 5. Horizon専用Walk-Forward Fold
# ============================================================

@dataclass(
    frozen=True
)
class HorizonFold:

    number: int

    train: pd.DataFrame

    core: pd.DataFrame

    validation: pd.DataFrame

    test: pd.DataFrame


def horizon_outer_folds(
    data,
    horizon_bars,
):

    """
    過去 → 未来の順番を絶対に崩さない。

    horizon_bars分のgapを入れて、
    未来ラベルが次区間へ跨がないようにする。
    """

    block = (
        len(data)
        //
        (
            OUTER_SPLITS_HORIZON
            + 1
        )
    )

    if block < 1:
        return

    for number in range(
        1,
        OUTER_SPLITS_HORIZON + 1
    ):

        train_end = (
            block
            * number
        )

        # Testとの間にgap
        test_start = (
            train_end
            +
            horizon_bars
        )

        test_end = min(
            test_start
            +
            block,
            len(data)
        )

        train = (
            data.iloc[
                :train_end
            ]
        )

        if len(train) < 100:
            continue

        # Trainの後ろ20%をValidation
        cut = int(
            len(train)
            * 0.80
        )

        core_end = max(
            0,
            cut
            -
            horizon_bars
        )

        core = (
            train.iloc[
                :core_end
            ]
        )

        validation = (
            train.iloc[
                cut:
            ]
        )

        test = (
            data.iloc[
                test_start:test_end
            ]
        )

        if (
            len(core) == 0
            or
            len(validation) == 0
            or
            len(test) == 0
        ):
            continue

        # ----------------------------
        # Leakage check
        # ----------------------------

        if not (
            core[
                "label_end"
            ]
            <=
            validation.index[0]
        ).all():

            raise ValueError(
                "Core label crosses Validation boundary"
            )

        if not (
            train[
                "label_end"
            ]
            <=
            test.index[0]
        ).all():

            raise ValueError(
                "Train label crosses Test boundary"
            )

        yield HorizonFold(
            number=
                number,

            train=
                train,

            core=
                core,

            validation=
                validation,

            test=
                test,
        )


# ============================================================
# 6. Directionモデル
# ============================================================

def fit_horizon_direction_model(
    train,
    trees=350,
):

    target = (
        train[
            "direction_target"
        ]
    )

    if (
        target.nunique()
        < 2
    ):

        return None

    weights = make_time_weights(
        train.index,
        HALF_LIFE_DAYS
    )

    model = RandomForestClassifier(

        n_estimators=
            trees,

        max_depth=
            8,

        min_samples_leaf=
            20,

        max_features=
            "sqrt",

        class_weight=
            "balanced",

        random_state=
            RANDOM_STATE,

        n_jobs=
            -1,
    )

    model.fit(

        train[
            direction_features
        ],

        target,

        sample_weight=
            weights,
    )

    return model


# ============================================================
# 7. P(up) / P(down)
# ============================================================

def predict_horizon_direction(
    model,
    frame,
):

    probability = (
        model.predict_proba(
            frame[
                direction_features
            ]
        )
    )

    class_map = {
        c: i
        for i, c
        in enumerate(
            model.classes_
        )
    }

    p_down = (
        probability[
            :,
            class_map[0]
        ]
    )

    p_up = (
        probability[
            :,
            class_map[1]
        ]
    )

    return (
        p_up,
        p_down
    )


# ============================================================
# 8. 時間決済シグナル
# ============================================================

def make_horizon_signals(
    p_up,
    p_down,
    threshold,
):

    signals = np.zeros(
        len(p_up)
    )

    buy = (
        (p_up >= threshold)
        &
        (p_up > p_down)
    )

    sell = (
        (p_down >= threshold)
        &
        (p_down > p_up)
    )

    signals[
        buy
    ] = 1

    signals[
        sell
    ] = -1

    return signals


# ============================================================
# 9. 時間決済Backtest
# ============================================================

def run_horizon_time_exit_backtest(
    frame,
    signals,
    horizon_minutes,
    *,
    cost=TRADING_COST,
):

    """
    TP/SLなし。

    BUY:
        future_return - cost

    SELL:
        -future_return - cost

    これにより
    「ホライズンそのもの」を比較する。
    """

    records = []

    horizon_bars = (
        horizon_minutes
        //
        BAR_MINUTES
    )

    next_allowed_position = -1

    for i, (
        time,
        signal
    ) in enumerate(
        zip(
            frame.index,
            signals
        )
    ):

        if signal == 0:
            continue

        # frame内位置で重複取引防止
        if (
            i
            <
            next_allowed_position
        ):
            continue

        future_return = float(
            frame[
                "future_return"
            ].iloc[i]
        )

        if signal == 1:

            gross = (
                future_return
            )

            side = "BUY"

            confidence = np.nan

        else:

            gross = (
                -future_return
            )

            side = "SELL"

            confidence = np.nan

        net = (
            gross
            -
            cost
        )

        records.append(
            {
                "signal_time":
                    time,

                "direction":
                    side,

                "gross_return":
                    gross,

                "cost":
                    cost,

                "net_return":
                    net,

                "actual_future_return":
                    future_return,
            }
        )

        # ----------------------------
        # 1ポジションずつ
        # ----------------------------

        next_allowed_position = (
            i
            +
            horizon_bars
        )

    return pd.DataFrame(
        records
    )


# ============================================================
# 10. 統計
# ============================================================

def horizon_stats(
    returns,
):

    r = np.asarray(
        returns,
        dtype=float
    )

    if len(r) == 0:

        return {
            "trades":
                0,

            "win_rate":
                np.nan,

            "avg_return":
                np.nan,

            "profit_factor":
                np.nan,

            "max_dd":
                np.nan,

            "total_growth":
                0.0,

            "sharpe":
                np.nan,
        }

    gains = (
        r[
            r > 0
        ].sum()
    )

    losses = (
        -r[
            r < 0
        ].sum()
    )

    if losses > 0:

        pf = (
            gains
            /
            losses
        )

    elif gains > 0:

        pf = np.inf

    else:

        pf = np.nan

    equity = np.r_[
        1.0,
        np.cumprod(
            1 + r
        )
    ]

    dd = (
        equity
        /
        np.maximum.accumulate(
            equity
        )
        - 1
    )

    if (
        len(r) > 1
        and
        np.std(r) > 0
    ):

        sharpe = (
            np.mean(r)
            /
            np.std(r)
            *
            np.sqrt(
                len(r)
            )
        )

    else:

        sharpe = np.nan

    return {
        "trades":
            len(r),

        "win_rate":
            (
                r > 0
            ).mean(),

        "avg_return":
            r.mean(),

        "profit_factor":
            pf,

        "max_dd":
            dd.min(),

        "total_growth":
            equity[-1]
            - 1,

        "sharpe":
            sharpe,
    }


# ============================================================
# 11. ValidationでConfidence thresholdを選ぶ
# ============================================================

def choose_horizon_threshold(
    validation,
    p_up,
    p_down,
    horizon_minutes,
):

    best = None

    best_score = -np.inf

    for threshold in (
        CONFIDENCE_THRESHOLDS
    ):

        signals = (
            make_horizon_signals(
                p_up,
                p_down,
                threshold
            )
        )

        trades = (
            run_horizon_time_exit_backtest(
                validation,
                signals,
                horizon_minutes
            )
        )

        if (
            len(trades)
            <
            MIN_VALIDATION_TRADES_HORIZON
        ):
            continue

        stats = horizon_stats(
            trades[
                "net_return"
            ]
        )

        # ----------------------------
        # 平均利益 × sqrt(N)
        #
        # 少数取引だけで極端な値が
        # 選ばれにくくする
        # ----------------------------

        score = (
            stats[
                "avg_return"
            ]
            *
            np.sqrt(
                stats[
                    "trades"
                ]
            )
        )

        if (
            score
            >
            best_score
        ):

            best_score = score

            best = {
                "threshold":
                    threshold,

                "validation_trades":
                    stats[
                        "trades"
                    ],

                "validation_avg_return":
                    stats[
                        "avg_return"
                    ],

                "validation_pf":
                    stats[
                        "profit_factor"
                    ],

                "score":
                    score,
            }

    return best


# ============================================================
# 12. 全Horizon実験
# ============================================================

all_horizon_rows = []

all_fold_rows = []

all_trade_frames = []


for horizon_minutes in (
    HORIZONS_MINUTES
):

    print()
    print(
        "########################################"
    )

    print(
        f"{horizon_minutes} MIN MODEL"
    )

    print(
        "########################################"
    )

    horizon_bars = (
        horizon_minutes
        //
        BAR_MINUTES
    )

    horizon_data = (
        prepare_horizon_data(
            bars,
            horizon_minutes
        )
    )

    print(
        "使用可能データ:",
        len(
            horizon_data
        )
    )

    horizon_trade_frames = []

    evaluated_folds = 0

    positive_folds = 0

    pf_positive_folds = 0

    auc_values = []

    accuracy_values = []

    selected_thresholds = []

    for fold in horizon_outer_folds(
        horizon_data,
        horizon_bars
    ):

        print()
        print(
            f"Fold {fold.number}"
        )

        if (
            len(
                fold.core
            )
            < 300
        ):

            print(
                "学習データ不足でskip"
            )

            continue

        # ====================================================
        # Validationモデル
        # ====================================================

        validation_model = (
            fit_horizon_direction_model(
                fold.core,
                trees=300
            )
        )

        if (
            validation_model
            is None
        ):

            continue

        (
            val_p_up,
            val_p_down
        ) = (
            predict_horizon_direction(
                validation_model,
                fold.validation
            )
        )

        setting = (
            choose_horizon_threshold(
                fold.validation,
                val_p_up,
                val_p_down,
                horizon_minutes
            )
        )

        if setting is None:

            print(
                "Validation設定なし"
            )

            continue

        print(
            "選択Threshold:",
            setting[
                "threshold"
            ]
        )

        selected_thresholds.append(
            setting[
                "threshold"
            ]
        )

        # ====================================================
        # Test直前まで再学習
        # ====================================================

        final_model = (
            fit_horizon_direction_model(
                fold.train,
                trees=400
            )
        )

        if (
            final_model
            is None
        ):

            continue

        (
            test_p_up,
            test_p_down
        ) = (
            predict_horizon_direction(
                final_model,
                fold.test
            )
        )

        # ====================================================
        # 純粋なDirection性能
        # ====================================================

        actual = (
            fold.test[
                "direction_target"
            ].values
        )

        try:

            auc = (
                roc_auc_score(
                    actual,
                    test_p_up
                )
            )

        except:

            auc = np.nan

        prediction = (
            test_p_up
            >
            test_p_down
        ).astype(int)

        accuracy = (
            accuracy_score(
                actual,
                prediction
            )
        )

        auc_values.append(
            auc
        )

        accuracy_values.append(
            accuracy
        )

        # ====================================================
        # Test取引
        # ====================================================

        signals = (
            make_horizon_signals(
                test_p_up,
                test_p_down,
                setting[
                    "threshold"
                ]
            )
        )

        trades = (
            run_horizon_time_exit_backtest(
                fold.test,
                signals,
                horizon_minutes
            )
        )

        if (
            not trades.empty
        ):

            trades[
                "fold"
            ] = fold.number

            trades[
                "horizon_minutes"
            ] = horizon_minutes

            horizon_trade_frames.append(
                trades
            )

            all_trade_frames.append(
                trades
            )

        stats = horizon_stats(
            trades[
                "net_return"
            ]
            if not trades.empty
            else []
        )

        evaluated_folds += 1

        if (
            stats[
                "avg_return"
            ]
            > 0
        ):

            positive_folds += 1

        if (
            stats[
                "profit_factor"
            ]
            > 1
        ):

            pf_positive_folds += 1

        fold_row = {
            "horizon_minutes":
                horizon_minutes,

            "fold":
                fold.number,

            "threshold":
                setting[
                    "threshold"
                ],

            "auc":
                auc,

            "accuracy":
                accuracy,

            **stats,
        }

        all_fold_rows.append(
            fold_row
        )

        print(
            "Trades:",
            stats[
                "trades"
            ],

            "Win:",
            round(
                stats[
                    "win_rate"
                ]
                * 100,
                2
            )
            if not np.isnan(
                stats[
                    "win_rate"
                ]
            )
            else np.nan,

            "Avg:",
            round(
                stats[
                    "avg_return"
                ]
                * 100,
                5
            )
            if not np.isnan(
                stats[
                    "avg_return"
                ]
            )
            else np.nan,

            "%",

            "PF:",
            round(
                stats[
                    "profit_factor"
                ],
                3
            )
            if np.isfinite(
                stats[
                    "profit_factor"
                ]
            )
            else stats[
                "profit_factor"
            ],
        )

    # ========================================================
    # Horizon総合
    # ========================================================

    if (
        horizon_trade_frames
    ):

        horizon_trades = (
            pd.concat(
                horizon_trade_frames,
                ignore_index=True
            )
        )

        total_stats = (
            horizon_stats(
                horizon_trades[
                    "net_return"
                ]
            )
        )

    else:

        horizon_trades = (
            pd.DataFrame()
        )

        total_stats = (
            horizon_stats(
                []
            )
        )

    row = {
        "horizon_minutes":
            horizon_minutes,

        "evaluated_folds":
            evaluated_folds,

        "positive_folds":
            positive_folds,

        "pf_above_1_folds":
            pf_positive_folds,

        "mean_auc":
            (
                np.nanmean(
                    auc_values
                )
                if len(
                    auc_values
                )
                else np.nan
            ),

        "mean_accuracy":
            (
                np.nanmean(
                    accuracy_values
                )
                if len(
                    accuracy_values
                )
                else np.nan
            ),

        "mean_selected_threshold":
            (
                np.mean(
                    selected_thresholds
                )
                if len(
                    selected_thresholds
                )
                else np.nan
            ),

        **total_stats,
    }

    all_horizon_rows.append(
        row
    )


# ============================================================
# 13. 結果表
# ============================================================

results = pd.DataFrame(
    all_horizon_rows
)

fold_results = pd.DataFrame(
    all_fold_rows
)


# ============================================================
# 14. %表示用
# ============================================================

shown = (
    results.copy()
)

for col in [
    "mean_accuracy",
    "win_rate",
    "avg_return",
    "max_dd",
    "total_growth",
]:

    shown[
        col
    ] = (
        shown[
            col
        ]
        * 100
    )


print()
print(
    "=========================================="
)

print(
    "HORIZON 最終比較"
)

print(
    "=========================================="
)


display_columns = [
    "horizon_minutes",
    "evaluated_folds",
    "positive_folds",
    "pf_above_1_folds",
    "trades",
    "mean_auc",
    "mean_accuracy",
    "win_rate",
    "avg_return",
    "profit_factor",
    "max_dd",
    "total_growth",
    "sharpe",
]


print(
    shown[
        display_columns
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 15. PFランキング
# ============================================================

print()
print(
    "=========================================="
)

print(
    "Profit Factorランキング"
)

print(
    "=========================================="
)


pf_ranking = (
    shown.sort_values(
        "profit_factor",
        ascending=False
    )
)


print(
    pf_ranking[
        [
            "horizon_minutes",
            "trades",
            "win_rate",
            "avg_return",
            "profit_factor",
            "total_growth",
            "positive_folds",
            "evaluated_folds",
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 16. 平均Returnランキング
# ============================================================

print()
print(
    "=========================================="
)

print(
    "平均リターンランキング"
)

print(
    "=========================================="
)


return_ranking = (
    shown.sort_values(
        "avg_return",
        ascending=False
    )
)


print(
    return_ranking[
        [
            "horizon_minutes",
            "trades",
            "avg_return",
            "profit_factor",
            "total_growth",
            "sharpe",
        ]
    ]
    .to_string(
        index=False
    )
)


# ============================================================
# 17. 30分との比較
# ============================================================

print()
print(
    "=========================================="
)

print(
    "30分モデル比較"
)

print(
    "=========================================="
)


thirty = (
    shown.loc[
        shown[
            "horizon_minutes"
        ]
        == 30
    ]
)


if (
    not thirty.empty
):

    thirty_pf = (
        thirty[
            "profit_factor"
        ].iloc[0]
    )

    thirty_return = (
        thirty[
            "avg_return"
        ].iloc[0]
    )

    better_pf = (
        shown.loc[
            shown[
                "profit_factor"
            ]
            >
            thirty_pf,
            "horizon_minutes"
        ]
        .tolist()
    )

    better_return = (
        shown.loc[
            shown[
                "avg_return"
            ]
            >
            thirty_return,
            "horizon_minutes"
        ]
        .tolist()
    )

    print(
        "30分 PF:",
        thirty_pf
    )

    print(
        "30分 平均Return:",
        thirty_return,
        "%"
    )

    print()

    print(
        "30分よりPFが高いHorizon:",
        better_pf
    )

    print(
        "30分より平均Returnが高いHorizon:",
        better_return
    )


# ============================================================
# 18. グラフ
# Horizon vs Profit Factor
# ============================================================

plt.figure(
    figsize=(
        9,
        5
    )
)

plt.plot(
    results[
        "horizon_minutes"
    ],
    results[
        "profit_factor"
    ],
    marker="o"
)

plt.axhline(
    1,
    linewidth=1
)

plt.xlabel(
    "Forecast horizon (minutes)"
)

plt.ylabel(
    "Profit Factor"
)

plt.title(
    "Forecast Horizon vs Profit Factor"
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()

plt.show()


# ============================================================
# 19. Horizon vs Average Return
# ============================================================

plt.figure(
    figsize=(
        9,
        5
    )
)

plt.plot(
    results[
        "horizon_minutes"
    ],
    results[
        "avg_return"
    ]
    * 100,
    marker="o"
)

plt.axhline(
    0,
    linewidth=1
)

plt.xlabel(
    "Forecast horizon (minutes)"
)

plt.ylabel(
    "Average net return / trade (%)"
)

plt.title(
    "Forecast Horizon vs Average Return"
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()

plt.show()


# ============================================================
# 20. Horizon vs AUC
# ============================================================

plt.figure(
    figsize=(
        9,
        5
    )
)

plt.plot(
    results[
        "horizon_minutes"
    ],
    results[
        "mean_auc"
    ],
    marker="o"
)

plt.axhline(
    0.5,
    linewidth=1
)

plt.xlabel(
    "Forecast horizon (minutes)"
)

plt.ylabel(
    "Direction ROC-AUC"
)

plt.title(
    "Forecast Horizon vs Direction AUC"
)

plt.grid(
    alpha=0.25
)

plt.tight_layout()

plt.show()


# ============================================================
# 21. 保存
# ============================================================

output_dir = (
    Path.cwd()
    /
    "horizon_comparison_experiment"
)

output_dir.mkdir(
    exist_ok=True
)


results.to_csv(
    output_dir
    / "horizon_summary.csv",
    index=False
)

fold_results.to_csv(
    output_dir
    / "horizon_fold_results.csv",
    index=False
)

if (
    all_trade_frames
):

    pd.concat(
        all_trade_frames,
        ignore_index=True
    ).to_csv(
        output_dir
        / "all_horizon_trades.csv",
        index=False
    )


print()
print(
    "=========================================="
)

print(
    "実験完了"
)

print(
    "=========================================="
)

print(
    "保存先:"
)

print(
    output_dir.resolve()
)

print()
print(
    "次に見る数字:"
)

print(
    "1. PFが最大のHorizon"
)

print(
    "2. 平均Returnが最大のHorizon"
)

print(
    "3. AUCが最大のHorizon"
)

print(
    "4. プラスFold数"
)

print(
    "5. 30分周辺が本当にピークになっているか"
)